In [1]:
# the goal: check distribution of FE and Q to select good thresholds
# the problem: the peaks defined in consensus are not at the same exact location as in in .narrowPeaks files. Therefore the matching is not straighforward

# approach
# get the idr consensus peaks
# select the max signal and q value across summits
# then append the values for each replicate
# calculate the median across replicates: that would be the position of the point
# gold orange or red will be if it passes thresholds across one two or three replicates
# typical stringent thresholds would be (AI says)
# FE ≥ 4 and qValue ≥ 5 (≈ FDR ≤ 10⁻⁵)

# make the same figure but with viridis on the length to see if the patterns are good of TF binding site

In [2]:
import pandas, pyranges, pathlib

In [3]:
# paths
base = pathlib.Path("/Users/adrian/research/bmcbf/004_keilir/results/1023_idr_results/MITF_A")

consensus_file = base / "MITF_A.IDR_consensus.bed"

idr_files = [
    base / "idr_r1_r2.txt",
    base / "idr_r1_r3.txt",
    base / "idr_r2_r3.txt",
]

narrow_files = [
    base / "rep1.idr.sorted.narrowPeak",
    base / "rep2.idr.sorted.narrowPeak",
    base / "rep3.idr.sorted.narrowPeak",
]

In [5]:
# load consensus peaks
consensus = pandas.read_csv(consensus_file, sep="\t", header=None, usecols=[0,1,2], names=["chromosome","start","end"])
print(consensus.shape)
consensus = consensus.set_index(consensus["chromosome"].astype(str) + ":" + consensus["start"].astype(str) + "-" + consensus["end"].astype(str))
consensus

(41664, 3)


,chromosome,start,end
chr1:910547-911772,chr1,910547,911772
chr1:912004-912799,chr1,912004,912799
chr1:921531-924309,chr1,921531,924309
chr1:930276-932047,chr1,930276,932047
chr1:936850-940044,chr1,936850,940044
...,...,...,...
chrX:154762416-154764110,chrX,154762416,154764110
chrX:154804348-154805746,chrX,154804348,154805746
chrX:155026633-155027562,chrX,155026633,155027562
chrX:155070754-155072190,chrX,155070754,155072190


In [29]:
intersect = df[(df["start"] == row.start) & (df["end"] == row.end)]

In [31]:
intersect.index

Index(['chr1:2135476-2136663'], dtype='object')

In [37]:
common_indexes = []

for idr_file in idr_files:
    
    df = pandas.read_csv(idr_file, sep='\t', header=None)
    df = df.rename(columns={df.columns[0]: "chromosome", df.columns[1]: "start", df.columns[2]: "end"})
    df = df.set_index(df["chromosome"].astype(str) + ":" + df["start"].astype(str) + "-" + df["end"].astype(str))

    # check that the names in idr files are the same as in consensus. it seems that is not the case. Check if the consensus index can be on the columns, but not in the inded of the idr.
    common_index = consensus.index.intersection(df.index)
    print(len(common_index))
    common_indexes.append(common_index)

union_index = common_indexes[0].union(common_indexes[1]).union(common_indexes[2])
print(len(union_index))


diff_index = consensus.index.difference(union_index)
diff_index

18707
3460
37442
40106


Index(['chr10:101123860-101125546', 'chr10:101221483-101222347',
       'chr10:101617970-101619156', 'chr10:101817375-101819372',
       'chr10:101841801-101843990', 'chr10:102117308-102120180',
       'chr10:102398973-102402818', 'chr10:103589093-103591357',
       'chr10:109884031-109884794', 'chr10:110037205-110037579',
       ...
       'chrX:153688870-153691967', 'chrX:154804348-154805746',
       'chrX:17769033-17770605', 'chrX:17771251-17772565',
       'chrX:30788510-30790277', 'chrX:48460630-48462324',
       'chrX:53262980-53264428', 'chrX:53292766-53293775',
       'chrX:56564141-56565707', 'chrX:70444354-70447266'],
      dtype='object', length=1558)

In [17]:
df

,chromosome,start,end,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
chr8:143929676-143947403,chr8,143929676,143947403,.,1000,.,962.18149,-1,-1,7666,5.000000,5.000000,143930364,143947195,166.73799,149,143929676,143947403,795.44350,8872
chr22:23105873-23120749,chr22,23105873,23120749,.,1000,.,957.83264,-1,-1,7459,5.000000,5.000000,23106083,23120426,230.29113,304,23105873,23120749,727.54151,7161
chr10:43248813-43260789,chr10,43248813,43260789,.,1000,.,777.64453,-1,-1,10822,5.000000,5.000000,43248901,43260717,204.37763,562,43248813,43260789,573.26690,5850
chr1:3493422-3506779,chr1,3493422,3506779,.,1000,.,759.12921,-1,-1,10724,5.000000,5.000000,3493498,3506607,115.74392,155,3493422,3506779,643.38529,6580
chr8:143377154-143389954,chr8,143377154,143389954,.,1000,.,752.75452,-1,-1,11794,5.000000,5.000000,143377253,143389909,162.29603,230,143377154,143389954,590.45849,6272
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
chr20:58196151-58196686,chr20,58196151,58196686,.,540,.,30.24427,-1,-1,258,0.666064,1.302368,58196151,58196656,8.11423,136,58196193,58196686,22.13004,263
chr16:81159433-81160056,chr16,81159433,81160056,.,540,.,30.23464,-1,-1,237,0.666337,1.302666,81159493,81159946,8.40787,20,81159433,81160056,21.82677,307
chr16:30797708-30798246,chr16,30797708,30798246,.,540,.,30.20873,-1,-1,241,0.665402,1.301476,30797817,30798098,8.22287,126,30797708,30798246,21.98586,233
chr18:46322404-46322905,chr18,46322404,46322905,.,540,.,30.20294,-1,-1,182,0.665739,1.301996,46322412,46322888,8.30841,20,46322404,46322905,21.89453,290


In [ ]:
d

In [ ]:
cge

In [ ]:
# narrowPeak = BED6+4: 0..9
rep_path = rep_files[0]
cols = ["chromosome","start","end","name","score","strand","FE","pval","qval","summit"]
rep_df = pandas.read_csv(rep_path, sep="\t", header=None, names=cols, dtype={"chromosome":str})
rep_df = rep_df[["Chromosome","Start","End","FE","qval"]]
print('\t', rep_df.shape)


In [ ]:
for each consensus peak, check the r1_r2 files and see if it is there

# read r1_r2, detect the coordinates, iterate idr files and check that positions are concordant per replicate
# theck that consensus and idr indexes match!!!! follow this thread!!!

In [ ]:
asdf

In [ ]:
# 2) Function to annotate one replicate by max FE/q over overlaps
def annotate_one_rep(cons_gr, rep_path, rep_idx):

    #print(rep_p
    
    # narrowPeak = BED6+4: 0..9
    cols = ["Chromosome","Start","End","name","score","strand","FE","pval","qval","summit"]
    rep_df = pandas.read_csv(rep_path, sep="\t", header=None, names=cols, dtype={"Chromosome":str})
    rep_df = rep_df[["Chromosome","Start","End","FE","qval"]]
    print('\t', rep_df.shape)
    rep_gr = pyranges.PyRanges(rep_df)
    print('\t success first block')

    # Overlap, then aggregate to consensus intervals: take max FE and max qval per consensus interval
    ov = cons_gr.join(rep_gr, how="left", suffix="_rep")  # keeps consensus intervals with overlapping rep intervals
    # ov.as_df() has Start, End from consensus plus FE, qval from rep where overlapping; may be NaN if no overlap
    ov_df = ov.as_df()[["Chromosome","Start","End","FE","qval"]]
    print('\t', ov_df.shape)
    print('\t success second block')
    
    # Group by consensus coord and take max (skip NaN)
    print('about to run last step')
    print(ov_df.shape)
    print(ov_df.memory_usage(deep=True).sum() / 1e9, "GB")
    print(ov_df.groupby(["Chromosome","Start","End"]).size().describe())
    print('print completed')
    agg = (ov_df.groupby(["Chromosome","Start","End"], as_index=False).agg({"FE":"max", "qval":"max"}))
    print('done')
    print()
    #agg = agg.rename(columns={"FE":f"FE_r{rep_idx}", "qval":f"q_r{rep_idx}"})
    return None

ann_list = []
for i, rep in enumerate(rep_files, start=1):
    print(i, rep)
    print(cons_gr)
    print('about to call function...')
    ann_list.append(annotate_one_rep(cons_gr, rep, i))
    
    print()

    #ann_list.append(annotate_one_rep(cons_gr, rep, i))

In [ ]:


# 2) Function to annotate one replicate by max FE/q over overlaps
def annotate_one_rep(cons_gr, rep_path, rep_idx):
    # narrowPeak = BED6+4: 0..9
    cols = ["Chromosome","Start","End","name","score","strand","FE","pval","qval","summit"]
    rep_df = pandas.read_csv(rep_path, sep="\t", header=None, names=cols, dtype={"Chromosome":str})
    rep_df = rep_df[["Chromosome","Start","End","FE","qval"]]
    rep_gr = pyranges.PyRanges(rep_df)

    # Overlap, then aggregate to consensus intervals: take max FE and max qval per consensus interval
    ov = cons_gr.join(rep_gr, how="left", suffix="_rep")  # keeps consensus intervals with overlapping rep intervals
    # ov.as_df() has Start, End from consensus plus FE, qval from rep where overlapping; may be NaN if no overlap
    ov_df = ov.as_df()[["Chromosome","Start","End","FE","qval"]]
    # Group by consensus coord and take max (skip NaN)
    agg = (ov_df
           .groupby(["Chromosome","Start","End"], as_index=False)
           .agg({"FE":"max", "qval":"max"}))
    agg = agg.rename(columns={"FE":f"FE_r{rep_idx}", "qval":f"q_r{rep_idx}"})
    return agg

ann_list = []
for i, rep in enumerate(rep_files, start=1):
    ann_list.append(annotate_one_rep(cons_gr, rep, i))

# 3) Merge per-replicate annotations back to consensus
merged = cons_df.copy()
for df in ann_list:
    merged = merged.merge(df, on=["Chromosome","Start","End"], how="left")

# 4) Median across replicates (skip NaN)
fe_cols = [c for c in merged.columns if c.startswith("FE_r")]
q_cols  = [c for c in merged.columns if c.startswith("q_r")]

merged["FE_median"] = merged[fe_cols].median(axis=1, skipna=True)
merged["q_median"]  = merged[q_cols].median(axis=1, skipna=True)

# 5) Save a minimal output
out = base / "MITF_A.IDR_consensus.with_FE_q.median.tsv"
merged[["Chromosome","Start","End","FE_median","q_median"]].to_csv(out, sep="\t", index=False)
print(f"Wrote: {out}")


In [ ]:
wd = '/Users/adrian/research/bmcbf/004_keilir/results/1023_idr_results/MITF_A/'
consensus_file = 'MITF_A.IDR_consensus.bed'

In [ ]:
con = pandas.read_csv(wd+consensus_file, sep='\t', header=None)
con.columns = ["chr", "start", "end"]
print(con.shape)

con = con.iloc[:100, :]

con['peak_id'] = con["chr"].astype(str) + ":" + con["start"].astype(str) + "-" + con["end"].astype(str)
con = con.set_index("peak_id")

con

In [ ]:
#  signal, P value, Q value and distance to summit. Signal is the same as fold enrichment in xls. not log2 FC, but just FC
df = pandas.read_csv(wd+'rep1.idr.sorted.narrowPeak', sep='\t', header=None)
df = df.rename(columns={0: 'chr', 1: 'start', 2:'end', 6:'FE', 8:'qvalue'})
df['peak_id'] = df["chr"].astype(str) + ":" + df["start"].astype(str) + "-" + df["end"].astype(str)
df = df.set_index("peak_id")

df

In [ ]:
subset = df.loc[df.index.intersection(con.index)]
subset

In [ ]:
subset

In [ ]:
df